# Metaclasses – Practice Exercises

This notebook contains hands-on exercises to practice Python **metaclasses**.

The progression goes from basics (`type` and dynamic class creation) to practical patterns like registries, validation contracts, and a small declarative mini-DSL.

For each exercise:

- Read the description carefully.
- Implement the TODO parts in the provided code cell.
- Run the demo tests at the bottom of the cell.


### Contents

- [Exercise 1 – Dynamic Classes with `type`](#exercise-1--dynamic-classes-with-type)
- [Exercise 2 – Logging Metaclass](#exercise-2--logging-metaclass)
- [Exercise 3 – Auto-Register Subclasses](#exercise-3--auto-register-subclasses)
- [Exercise 4 – Enforce Required Class Attributes](#exercise-4--enforce-required-class-attributes)
- [Exercise 5 – Declarative Mini-DSL](#exercise-5--declarative-mini-dsl)
- [Exercise 6 – Metaclass + ABC Contract](#exercise-6--metaclass--abc-contract)
- [Exercise 7 – Validate Instance Creation in `__call__`](#exercise-7--validate-instance-creation-in-__call__)
- [Exercise 8 – Singleton Metaclass](#exercise-8--singleton-metaclass)
- [Exercise 9 – Strategy Registry Use-Case](#exercise-9--strategy-registry-use-case)
- [Exercise 10 – When Not to Use Metaclasses](#exercise-10--when-not-to-use-metaclasses)


## Exercise 1 – Dynamic Classes with `type`

**Goal**: Use `type(name, bases, attrs)` to build a class dynamically.

**Requirements**:

- Create a dynamic class `DynamicOrder` inheriting from `object`.
- Give it a class attribute `kind = "dynamic"`.
- Instantiate it and verify `.kind` exists.

**Hint**: The second argument is a tuple of base classes.


In [15]:
# Exercise 1 – create DynamicOrder via type

DynamicOrder = type("DynamicOrder", (object,), {"kind": "dynamic"})
print(f"Class repr: \"{repr(DynamicOrder)}\"")
print(f"Class name: \"{DynamicOrder.__name__}\"")
print(f"Class type: \"{type(DynamicOrder)}\"")
ParentClass = DynamicOrder.__base__
print(f"Parent repr: \"{repr(ParentClass)}\"")
print(f"Parent name: \"{ParentClass.__name__}\"")
print(f"Parent name: \"{type(ParentClass)}\"")
print(f"Class kind: \"{DynamicOrder.kind}\"")

Class repr: "<class '__main__.DynamicOrder'>"
Class name: "DynamicOrder"
Class type: "<class 'type'>"
Parent repr: "<class 'object'>"
Parent name: "object"
Parent name: "<class 'type'>"
Class kind: "dynamic"


## Exercise 2 – Logging Metaclass

**Goal**: Intercept class creation with `__new__`.

**Requirements**:

- Create `LoggingMeta`.
- In `__new__`, print the class name being created.
- Create a base class `BaseModel(metaclass=LoggingMeta)`.
- Subclass it and verify the print happens.

**Hint**: `__new__(mcls, name, bases, namespace)` is called during class creation.


In [ ]:
# Exercise 2 – implement LoggingMeta

from typing import Any

class MetaClass(type):

    @classmethod
    def __prepare__(meta: type, name: str, bases: tuple):
        print("__" * 40, end = "\n\n")
        print(f"Input: \"__prepare__(\n\t{meta = },\n\t{name = },\n\t{bases = })\"...")
        attributes = super().__prepare__(name, bases)
        print(f"Output:\n\t\"{attributes = }\"")
        return attributes
        
    def __new__(meta, name, bases, attributes, **kwargs):
        print("__" * 40, end = "\n\n")
        print(f"Input: \"__new__(\n\t{meta = },\n\t{name = },\n\t{bases = },\n\t{attributes = })\"")
        _class = super().__new__(meta, name, bases, attributes, **kwargs)
        print(f"Output:\n\t\"{_class = }\"")
        attrs_str = str.join("\n", [f" - {repr(attr)}" for attr in attributes])
        bases_str = str.join("\n", [f" - {repr(base)}" for base in bases])
        if (len(bases) == 0): bases_str = "       N/A"
        print("_ " * 40, end = "\n\n")
        print(f"...Bases:\n{bases_str}\n...Attributes:\n{attrs_str}")
        return _class
        
    def __init__(cls, name: str, bases: tuple, attributes: dict, **kwargs):
        print("__" * 40, end = "\n\n")
        print(f"Input: \"__init__(\n\t{name = },\n\t{bases = },\n\t{attributes = })\"")
        _class = super().__init__(name, bases, attributes, **kwargs)
        print(f"Output:\n\t\"{_class = }\"")
        return _class

    def __call__(cls, *args, **kwargs):
        print("__" * 40, end = "\n\n")
        print(f"Input: \"__call__(\n\t{args = },\n\t{kwargs = })\"")
        result = super().__call__(*args, **kwargs)
        print(f"Output:\n\t\"{result = }\"")
        return result

class BaseClass(metaclass = MetaClass):
    """Base for classes using MetaClass."""
    arg1: Any = None
    arg2: Any = None
    def __repr__(self):
        return f"{self.arg1} {self.arg2}"

base_model = BaseClass()
print(f"\nInstance of \"{type(base_model).__name__}\": \"{repr(base_model)}\"")

## Exercise 3 – Auto-Register Subclasses

**Goal**: Build a registry of subclasses automatically.

**Requirements**:

- Create `RegistryMeta` metaclass.
- Maintain `registry: dict[str, type]` as a class attribute on the base class.
- Every subclass should define a unique `key` string.
- Register subclasses during `__init__` or `__new__`.
- Provide a helper `create(key, *args, **kwargs)` on the base class.

**Hint**: Look at `namespace` for `key`.


In [40]:
from typing import Any

class MetaClass(type):
    
    def __new__(meta: type, name: str, bases: tuple, attributes: dict, **kwargs):
        print(f"\nInput: \"__new__(\n\t{meta = },\n\t{name = },\n\t{bases = },\n\t{attributes = })\"")
        _class = super().__new__(meta, name, bases, attributes, **kwargs)
        registry: dict = getattr(_class, "_registry", None)
        if registry is not None:
            key = attributes.get("key", name)
            assert key not in registry
            registry[key] = _class
        else: setattr(_class, "_registry", dict[str, type]())
        print(f"Output:\n\t\"{_class = }\"")
        return _class

    def create(self, key: str, *args, **kwargs):
        registry: dict = getattr(self, "_registry", None)
        if registry is None: raise TypeError("This class is not base")
        if key in registry: return registry[key](*args, **kwargs)
        else: raise TypeError(f"\"{key}\" not found in base registry")

print("=" * 80)

class BaseClass(metaclass = MetaClass):
    _registry = dict[str, type]()
    def __init__(self, **kwargs):
        print(f"\nInstantiating \"BaseClass({kwargs = })\"")
        for key, value in kwargs.items():
            self.__setattr__(key, value)

class SubClass(BaseClass):
    key = "SubClass"
    pass

print("\n", "=" * 80)

bc = BaseClass(a = 1, b = 2)

print("\n", "=" * 80)

sc = BaseClass.create("SubClass")
print(sc)

print("\n", "=" * 80)


Input: "__new__(
	meta = <class '__main__.MetaClass'>,
	name = 'BaseClass',
	bases = (),
	attributes = {'__module__': '__main__', '__qualname__': 'BaseClass', '__firstlineno__': 25, '_registry': {}, '__init__': <function BaseClass.__init__ at 0x0000023BB44D0180>, '__static_attributes__': ()})"
Output:
	"_class = <class '__main__.BaseClass'>"

Input: "__new__(
	meta = <class '__main__.MetaClass'>,
	name = 'SubClass',
	bases = (<class '__main__.BaseClass'>,),
	attributes = {'__module__': '__main__', '__qualname__': 'SubClass', '__firstlineno__': 32, 'key': 'SubClass', '__static_attributes__': ()})"
Output:
	"_class = <class '__main__.SubClass'>"


Instantiating "BaseClass(kwargs = {'a': 1, 'b': 2})"


Instantiating "BaseClass(kwargs = {})"



## Exercise 4 – Enforce Required Class Attributes

**Goal**: Validate contracts at class creation time.

**Requirements**:

- Create `RequireKeyMeta` metaclass.
- For any subclass of `KeyedStrategy`, enforce that it defines `key`.
- If `key` is missing, raise `TypeError`.

**Hint**: Metaclass `__new__` can inspect the `namespace`.


In [41]:
# Exercise 4 – implement RequireKeyMeta
class MetaClass(type):
    
    def __new__(meta: type, name: str, bases: tuple, attributes: dict, **kwargs):
        _class = super().__new__(meta, name, bases, attributes, **kwargs)
        if bases and ((attr := "key") not in attributes):
            base = bases[0].__name__
            error = f"Class \"{name}\" is subclass of \"{base}\". "
            error += f"A \"{attr}\" class attribute must be defined."
            raise TypeError(error)
        return _class

class BaseClass(metaclass = MetaClass): pass

try:
    class SubClass1(BaseClass):
        key = "SubClass1"
        pass
    class SubClass2(BaseClass):
        #key = "SubClass2"
        pass

except Exception as EXC: print(repr(EXC))

TypeError('Class "SubClass2" is subclass of "BaseClass". A "key" class attribute must be defined.')


#### Another way...

In [42]:
# Exercise 4 – implement RequireKeyMeta
class MetaClass(type):
    
    def __new__(meta: type, name: str, bases: tuple, attributes: dict, **kwargs):
        _class = super().__new__(meta, name, bases, attributes, **kwargs)
        if bases and not hasattr(_class, attr := "key"):
            base = bases[0].__name__
            error = f"Class \"{name}\" is subclass of \"{base}\". "
            error += f"A \"{attr}\" class attribute must be defined."
            raise TypeError(error)
        return _class

class BaseClass(metaclass = MetaClass): pass

try:
    class SubClass1(BaseClass):
        key = "SubClass1"
        pass
    class SubClass2(BaseClass):
        #key = "SubClass2"
        pass

except Exception as EXC: print(repr(EXC))

TypeError('Class "SubClass2" is subclass of "BaseClass". A "key" class attribute must be defined.')


## Exercise 5 – Declarative Mini-DSL

**Goal**: Turn class-level declarations into runtime metadata.

**Requirements**:

- Create `Field` objects that store `name` and `default`.
- Create `DSLModelMeta` metaclass.
- For a subclass of `DSLModel`, collect all `Field` instances defined as class attributes into a dict `fields`.
- Create `DSLModel` base class using `DSLModelMeta`.
- Demonstrate with a `OrderSpec` model having 2-3 fields.

**Hint**: Iterate over `namespace.items()` in the metaclass.


In [87]:
# Exercise 5 – implement Field + DSLModelMeta
from typing import Any, AnyStr, Dict, Tuple, Type

class Field:
    def __init__(self, dtype: Type, default: Any = None):
        self.default, self.dtype = None, dtype
        if (default is not None):
            try: self.default = dtype(default)
            except: raise TypeError(f"Default value \"{default}\" " \
              f"should be compatible with type \"{dtype.__name__}\"")

class MetaClass(type):
    def __new__(meta: Type, name: AnyStr, bases: Tuple, attributes: Dict, **kwargs):
        fields: Dict[str, Field] = dict()
        for base in bases:
            attrs_base = getattr(base, "_fields", None)
            if attrs_base: fields.update(attrs_base)
        for attr_name, attr_value in attributes.copy().items():
            if isinstance(attr_value, Field):
                fields[attr_name] = attributes.pop(attr_name)
        attributes["_fields"] = fields
        return super().__new__(meta, name, bases, attributes, **kwargs)

class BaseClass(metaclass = MetaClass):
    _fields: Dict[str, Field]
    def __init__(self, **kwargs):
        for name, field in type(self)._fields.items():
            value = kwargs.get(name, field.default)
            if value is None: raise ValueError(
                f"Missing required field \"{name}\"")
            setattr(self, name, value)

    def __getattribute__(self, name: str):
        forbidden_attrs = {"_fields"}
        if (name not in forbidden_attrs): return object.__getattribute__(self, name)
        raise AttributeError(f"'{type(self).__name__}' object has no attribute '{name}'")        

class SubClass(BaseClass):
    int_field = Field(int)
    str_field = Field(str, "")
    num_field = Field(float, 0.0)

instance = SubClass(int_field = 12)

## Exercise 6 – Metaclass + ABC Contract

**Goal**: Combine metaclass behavior with ABC abstract methods.

**Requirements**:

- Create `BaseABC(ABC)` with an abstract `run()` method.
- Create a metaclass `CheckRunMeta` that checks a class defines `run` as an attribute.
- Create `class Constrained(BaseABC, metaclass=CheckRunMeta)`.
- Demonstrate:
- A subclass implementing `run()` works.
- A subclass missing `run` raises an error during class creation.

**Hint**: If you combine metaclasses, the metaclass must be compatible with ABC's metaclass behavior.


In [104]:
# Exercise 6 – implement CheckRunMeta + ABC interaction

from abc import ABC, abstractmethod, ABCMeta
from typing import Any, AnyStr, Dict, Tuple, Type

class MetaClass(ABCMeta):

    def __new__(meta: Type, name: AnyStr, bases: Tuple, attributes: Dict, **kwargs):
        _class = super().__new__(meta, name, bases, attributes, **kwargs)
        if not attributes.get("__abstract__", False):
            if not attributes.get("run", False):
                raise TypeError(f"\"{name}\" subclass needs \"run\" to be defined")
        
        return _class

class BaseClass(ABC, metaclass = MetaClass):
    __abstract__ = True
    @abstractmethod
    def run(self): ...

try:
    class SubClass1(BaseClass):
        def run(self): print("SubClass1 instance started")
    class SubClass2(BaseClass):
        def start(self): print("SubClass2 instance started")
except Exception as EXC: print(repr(EXC))

try: SubClass2().run()
except Exception as EXC: print(repr(EXC))

TypeError('"SubClass2" subclass needs "run" to be defined')
TypeError("Can't instantiate abstract class SubClass2 without an implementation for abstract method 'run'")


## Exercise 7 – Validate Instance Creation in `__call__`

**Goal**: Override metaclass `__call__` to validate constructor arguments.

**Requirements**:

- Create `NonNegativeMeta`.
- In `__call__`, if the class being instantiated receives `value=<...>`, require it to be non-negative.
- Instantiate a `Wrapper` class with `value` and show it works for valid input and fails otherwise.

**Hint**: `__call__` runs every time you create an instance.


In [139]:
# Exercise 7 – implement NonNegativeMeta

class MetaClass(type):

    def __call__(cls, *args, **kwargs):
        value = kwargs.get("value", None)
        assert (value >= 0), f"\"{value = }\" < 0"
        instance = super().__call__(*args, **kwargs)
        return instance

class BaseClass(metaclass = MetaClass):
    def __init__(self, value: float):
        self.value = value

try: right = BaseClass(value = +1)
except Exception as EXC: print(repr(EXC))
try: wrong = BaseClass(value = -1)
except Exception as EXC: print(repr(EXC))

AssertionError('"value = -1" < 0')


## Exercise 8 – Singleton Metaclass

**Goal**: Ensure only one instance exists per class.

**Requirements**:

- Implement `SingletonMeta`.
- Using it, create `class Config(metaclass=SingletonMeta)`.
- Instantiate `Config()` twice and show they are the same object (`is`).

**Hint**: Cache instances in a dict on the metaclass.


In [136]:
# Exercise 8 – implement SingletonMeta
from typing import Any, AnyStr, Dict, Tuple, Type, List

class MetaClass(type):

    _force_singleton = True

    def __new__(meta: Type, name: AnyStr, bases: Tuple, attributes: Dict, **kwargs):
        _class = super().__new__(meta, name, bases, attributes, **kwargs)
        setattr(_class, "_instances", list())
        return _class

    def __call__(cls, *args, **kwargs):
        instances: List = getattr(cls, "_instances")
        if instances and cls._force_singleton: return instances[0]
        instances.append(instance := super().__call__(*args, **kwargs))
        return instance

class BaseClass(metaclass = MetaClass): pass

b1 = BaseClass()
b2 = BaseClass()
b1, b2, b1 is b2

(<__main__.BaseClass at 0x23bb62e7230>,
 True)

## Exercise 9 – Strategy Registry Use-Case

**Goal**: Put Exercises 3/4 together into a practical registry pattern.

**Requirements**:

- Define a base `Strategy` using a registry metaclass (like `RegistryMeta`).
- Implement two strategies with different `key` values.
- Add a `run_strategy(key, *args, **kwargs)` function that creates and runs the strategy.
- Demonstrate calling `run_strategy()` for both keys.

**Hint**: Keep strategy behavior trivial; focus on registry wiring.


In [16]:
# Exercise 9 – registry use-case

from abc import ABC, abstractmethod, ABCMeta
from typing import Any, AnyStr, Dict, List, ClassVar, Iterable

class MetaStrategy(ABCMeta):
    def __new__(meta: ClassVar, name: AnyStr, bases: Iterable, attributes: Dict, **kwargs):
        if "__abstract__" not in attributes: attributes["__abstract__"] = False
        _class = super().__new__(meta, name, bases, attributes, **kwargs)
        if not hasattr(_class, "_registry"):
            _class._registry = dict()
        _registry: Dict = _class._registry
        if not getattr(_class, "__abstract__", False):
            key = getattr(_class, "key")
            assert key not in _registry
            _registry[key] = _class
        return _class

class BaseStrategy(ABC, metaclass = MetaStrategy):
    __abstract__ = True
    @classmethod
    def run_strategy(cls, key, *args, **kwargs):
        _registry: Dict[str, ClassVar] = getattr(cls, "_registry")
        assert key in _registry, f"\"{key}\" not registered in \"{cls.__name__}\""
        return _registry[key](*args, **kwargs)

try:
    class Strategy1(BaseStrategy): key = "Strategy1"
    class Strategy2(BaseStrategy): pass
except Exception as EXC: print(repr(EXC))

try:
    BaseStrategy.run_strategy("Strategy1")
    BaseStrategy.run_strategy("Strategy2")
except Exception as EXC: print(repr(EXC))

AttributeError("type object 'Strategy2' has no attribute 'key'")
AssertionError('"Strategy2" not registered in "BaseStrategy"')


## Exercise 10 – When Not to Use Metaclasses

**Goal**: Practice identifying an unnecessary metaclass pattern.

**Requirements**:

- Implement the same behavior from Exercise 8 (singleton) **without** metaclasses, using a closure or a decorator.
- Compare the two approaches in a short comment: what do you gain/lose?
- Demo: create `Config2` using the non-metaclass approach; show that `Config2()` is singleton.

**Hint**: A decorator that caches the instance is often enough.


In [ ]:
# Exercise 10 – singleton without metaclasses
from typing import Any, AnyStr, Dict, Tuple, Type, List, Callable

def singleton(n: int = 1):
    def wrapper1(init: Callable, *args, **kwargs):
        instances = list()
        def wrapper2(*args, **kwargs):
            if (len(instances) < n):
                instance = init(*args, **kwargs)
                instances.append(instance)
            return instances[-1]
        return wrapper2
    return wrapper1

@singleton(1)
class Singleton: pass

s1 = Singleton()
s2 = Singleton()
s1, s2, s1 is s2

(<__main__.Singleton at 0x2345c3dde80>,
 True)

: 